# DocBrain GPU Inference Server
**Run this on Google Colab Pro (GPU runtime)**

This notebook launches a FastAPI inference server with:
- **Donut** — OCR-free document parsing (GPU)
- **BGE embeddings** — fast vector generation (GPU)

The server is exposed via **ngrok** so your local FastAPI backend can call it.

### Setup
1. Runtime → Change runtime type → **T4 GPU**
2. Add your ngrok token to Colab Secrets (key: `NGROK_TOKEN`)
3. Run all cells — copy the printed URL into your local `backend/.env` as `COLAB_URL`

In [ ]:
# Cell 1: Install dependencies
!pip install -q fastapi uvicorn pyngrok Pillow transformers torch torchvision \
    sentence-transformers accelerate pdf2image pytesseract pdfplumber nest_asyncio
!apt-get install -q -y tesseract-ocr poppler-utils
print('Dependencies installed.')

In [ ]:
# Cell 2: Verify GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3: Load models (runs once, stays in memory)
from transformers import DonutProcessor, VisionEncoderDecoderModel
from sentence_transformers import SentenceTransformer

print('Loading Donut model...')
DONUT_MODEL_ID = 'naver-clova-ix/donut-base-finetuned-cord-v2'
donut_processor = DonutProcessor.from_pretrained(DONUT_MODEL_ID)
donut_model = VisionEncoderDecoderModel.from_pretrained(DONUT_MODEL_ID)
donut_model.to(device)
donut_model.eval()
print('Donut loaded.')

print('Loading BGE embedding model...')
BGE_MODEL_ID = 'BAAI/bge-small-en-v1.5'
embed_model = SentenceTransformer(BGE_MODEL_ID, device=device)
print('BGE loaded.')

print('All models ready.')

In [ ]:
# Cell 4: Define the FastAPI inference server
import io
import re
import base64
import json
import torch
import pytesseract
import pdfplumber
from PIL import Image
from pdf2image import convert_from_bytes
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional, Tuple

app = FastAPI(title='DocBrain GPU Inference Server')


# ── Request / Response models ─────────────────────────────────────────────────

class ParseRequest(BaseModel):
    file_b64: str          # base64-encoded file bytes
    filename: str          # used to detect extension
    use_donut: bool = True # if False, skip Donut and use Tesseract only

class ParseResponse(BaseModel):
    raw_text: str
    donut_json: Optional[dict] = None
    method: str            # 'donut', 'tesseract', 'pdfplumber'

class EmbedRequest(BaseModel):
    texts: List[str]

class EmbedResponse(BaseModel):
    embeddings: List[List[float]]
    model: str


# ── Helpers ───────────────────────────────────────────────────────────────────

def donut_parse_image(image: Image.Image) -> Tuple[str, dict]:
    """Run Donut on a PIL image. Returns (text, structured_json)."""
    task_prompt = '<s_cord-v2>'
    decoder_input_ids = donut_processor.tokenizer(
        task_prompt, add_special_tokens=False, return_tensors='pt'
    ).input_ids.to(device)

    pixel_values = donut_processor(
        image, return_tensors='pt'
    ).pixel_values.to(device)

    with torch.no_grad():
        outputs = donut_model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=donut_model.decoder.config.max_position_embeddings,
            early_stopping=True,
            pad_token_id=donut_processor.tokenizer.pad_token_id,
            eos_token_id=donut_processor.tokenizer.eos_token_id,
            use_cache=True,
            num_beams=1,
            bad_words_ids=[[donut_processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )

    sequence = donut_processor.batch_decode(outputs.sequences)[0]
    sequence = sequence.replace(donut_processor.tokenizer.eos_token, '')
    sequence = sequence.replace(donut_processor.tokenizer.pad_token, '')
    sequence = re.sub(r'<.*?>', '', sequence, count=1).strip()

    parsed = donut_processor.token2json(sequence)
    text = json.dumps(parsed, indent=2)
    return text, parsed


def tesseract_image(image: Image.Image) -> str:
    return pytesseract.image_to_string(image).strip()


def extract_pdf_text(file_bytes: bytes) -> Tuple[str, str]:
    """Try pdfplumber first; fall back to Tesseract on scanned pages."""
    text = ''
    with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + '\n'

    if text.strip():
        return text.strip(), 'pdfplumber'

    pages = convert_from_bytes(file_bytes, dpi=300)
    ocr_text = '\n'.join(tesseract_image(p) for p in pages)
    return ocr_text.strip(), 'tesseract'


# ── Endpoints ─────────────────────────────────────────────────────────────────

@app.get('/health')
def health():
    return {'status': 'ok', 'device': device}


@app.post('/parse', response_model=ParseResponse)
def parse_document(req: ParseRequest):
    file_bytes = base64.b64decode(req.file_b64)
    ext = req.filename.rsplit('.', 1)[-1].lower()

    if ext == 'pdf':
        raw_text, method = extract_pdf_text(file_bytes)
        return ParseResponse(raw_text=raw_text, method=method)

    image = Image.open(io.BytesIO(file_bytes)).convert('RGB')

    if req.use_donut:
        try:
            raw_text, donut_json = donut_parse_image(image)
            return ParseResponse(raw_text=raw_text, donut_json=donut_json, method='donut')
        except Exception as e:
            print(f'Donut failed ({e}), falling back to Tesseract')

    raw_text = tesseract_image(image)
    return ParseResponse(raw_text=raw_text, method='tesseract')


@app.post('/embed', response_model=EmbedResponse)
def embed_texts(req: EmbedRequest):
    if not req.texts:
        raise HTTPException(status_code=400, detail='texts list is empty')
    embeddings = embed_model.encode(
        req.texts, normalize_embeddings=True, show_progress_bar=False
    ).tolist()
    return EmbedResponse(embeddings=embeddings, model=BGE_MODEL_ID)


print('Server defined.')

In [ ]:
# Cell 5: Start ngrok tunnel and run server
import nest_asyncio
import uvicorn
import threading
from pyngrok import ngrok, conf
from google.colab import userdata

nest_asyncio.apply()

NGROK_TOKEN = userdata.get('NGROK_TOKEN')
conf.get_default().auth_token = NGROK_TOKEN

# Kill any existing tunnels
ngrok.kill()

# Open tunnel on port 8001
public_url = ngrok.connect(8001).public_url
print('\n' + '='*60)
print(f'  DocBrain GPU Server: {public_url}')
print('='*60)
print(f'  Add to backend/.env:  COLAB_URL={public_url}')
print('='*60 + '\n')

# Run server in a background thread so the cell doesn't block
def run():
    uvicorn.run(app, host='0.0.0.0', port=8001, log_level='warning')

t = threading.Thread(target=run, daemon=True)
t.start()
print('Server running. Keep this cell alive (do not interrupt).')

In [ ]:
# Cell 6 (Optional): Quick smoke test
import requests

r = requests.get(f'{public_url}/health')
print('Health check:', r.json())

# Test embedding
r = requests.post(f'{public_url}/embed', json={'texts': ['total amount due', 'invoice number']})
resp = r.json()
print(f'Embedding shape: {len(resp["embeddings"])} x {len(resp["embeddings"][0])}')
print(f'Model: {resp["model"]}')